# Tutorial: Möbius-Sattelzyklus - Periodenverdopplung und negative Eigenwerte

Dieses Notebook demonstriert einen topologisch interessanten Fixpunkt: einen **Möbius-Sattelzyklus**, bei dem die Monodromiematrix `DPm` **negative** Eigenwerte besitzt ($\lambda_u = -e^{+1/3}$, $\lambda_s = -e^{-1/3}$).

**Physikalische Bedeutung:**  
Nach einer vollständigen toroidalen Periode ($m=1$ Umlauf) wechselt eine kleine Verschiebung beim Strecken/Kontrahieren ihr *Vorzeichen*. Erst nach **zwei** Umläufen kehrt sie zu ihrer ursprünglichen Orientierung zurück - eine möbiusstreifenartige Topologie im lokalen Tangentialbündel.

Dies unterscheidet sich vom gewöhnlichen X-Punkt mit positiven Eigenwerten und wird manchmal als **invers-hyperbolischer** oder **periodenverdoppelter** Fixpunkt bezeichnet.

**Was Sie lernen:**
1. Wie Sattelpunkte mit negativen Eigenwerten in Poincaré-Karten entstehen
2. Wie `evolve_DPm_along_cycle` das Vorzeichen der Eigenwerte bestätigt
3. Wie die stabile/instabile Mannigfaltigkeitsgeometrie in 3-D visualisiert wird - sie bildet ein halb verdrehtes Band (Möbiusstreifen) um den Torus


## 1. Feldkonstruktion - m=1-Möbius-Fixpunkt bei (R₀, Z₀)

Der einfachste Fall: ein einzelner Fixpunkt der Ein-Umlauf-Karte bei $(R_0, Z_0)$ mit Eigenwerten $\lambda_u = -e^{+1/3}$ und $\lambda_s = -e^{-1/3}$.

Die Eigenrichtungen rotieren pro Umlauf um $\pi$ (`dθ/dφ = 1/2`), sodass nach einem Umlauf $\theta \to \theta + \pi$ gilt - die Eigenvektoren klappen um; daher der Möbius-Charakter.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# ── Fixpunkt and field parameters ────────────────────────────────────────
R0, Z0 = 1.0, 0.0
BPhi0 = 2.5
m = 1  # one-turn fixed point

# Negative eigenvalues  →  Möbius / inverse-hyperbolisch character
lam_u = -np.e ** (1/3)
lam_s = -np.e ** (-1/3)
print(f"λ_u = {lam_u:.6f}")
print(f"λ_s = {lam_s:.6f}")
print(f"λ_u · λ_s = {lam_u * lam_s:.10f}  (sollte sein +1 for flächenerhaltende Karte)")
print(f"Nach 2 Umläufen: λ_u² = {lam_u**2:.6f}  (jetzt positiv, normal hyperbolisch)")

# Eigendirections rotate by π per turn (half-integer winding)
dtheta_dphi = 0.5

def theta_u(phi):
    return np.pi/2 + phi/2

def theta_s(phi):
    return phi/2

# Note: theta_u(2π) = π/2 + π = 3π/2 ≠ theta_u(0)
# The eigenvectors have Umklappenped!  This is the Möbius twist.
print(f"\nθ_u(0) = {theta_u(0):.4f} rad = {np.degrees(theta_u(0)):.1f}°")
print(f"θ_u(2π) = {theta_u(2*np.pi):.4f} rad = {np.degrees(theta_u(2*np.pi)):.1f}°")
print(f"Differenz = {np.degrees(theta_u(2*np.pi) - theta_u(0)):.1f}°  (= 180° Umklappen → Möbius)")

λ_u = -1.395612
λ_s = -0.716531
λ_u · λ_s = 1.0000000000  (sollte sein +1 for flächenerhaltende Karte)
Nach 2 Umläufen: λ_u² = 1.947734  (jetzt positiv, normal hyperbolisch)

θ_u(0) = 1.5708 rad = 90.0°
θ_u(2π) = 4.7124 rad = 270.0°
Differenz = 180.0°  (= 180° Umklappen → Möbius)


In [2]:
# ── Toroidal field Modusl ─────────────────────────────────────────────────────
def BPhi(phi, RZ):
    """Toroidal field: R·B_φ = const."""
    return BPhi0 * R0 / RZ[0]

# ── A matrix at the fixed point ──────────────────────────────────────────────
def _eigvec_matrix(phi):
    return np.array([
        [np.cos(theta_u(phi)), np.cos(theta_s(phi))],
        [np.sin(theta_u(phi)), np.sin(theta_s(phi))],
    ])

def _A_matrix(phi):
    """Jacobian A(φ) of field direction g w.r.t. (R,Z) at the fixed point."""
    V = _eigvec_matrix(phi)
    # log(|λ|) / (2mπ) gives the per-radian growth rate
    mu_u = np.log(np.abs(lam_u)) / (2 * m * np.pi)
    mu_s = np.log(np.abs(lam_s)) / (2 * m * np.pi)
    Lam = np.diag([mu_u, mu_s])
    dV = np.array([
        [-dtheta_dphi * np.sin(theta_u(phi)), -dtheta_dphi * np.sin(theta_s(phi))],
        [ dtheta_dphi * np.cos(theta_u(phi)),  dtheta_dphi * np.cos(theta_s(phi))],
    ])
    return V @ Lam @ np.linalg.inv(V) + dV @ np.linalg.inv(V)

# ── φ-parameterised field direction g = R·B_pol/B_φ ─────────────────────────
def field_g_pol(phi, X_pol):
    """Field direction in (R,Z): g = A(φ) · (X - X_c) + 0 (fixed point has dX_c/dφ=0)."""
    Xc = np.array([R0, Z0])
    A = _A_matrix(phi)
    return A @ (X_pol - Xc)

# Verify: g on the fixed point is zero (it's a FIXED point, not just periodic orbit)
g0 = field_g_pol(0.5, np.array([R0, Z0]))
print("g am Fixpunkt (R0,Z0):", g0, "  (sollte sein 0)")

# ── pyna-compatible field function ───────────────────────────────────────────
def pyna_field(rzphi):
    R, Z, phi = rzphi[0], rzphi[1], rzphi[2]
    X_pol = np.array([R, Z])
    g = field_g_pol(phi, X_pol)
    # Regularise near the fixed point to avoid division by zero
    gnorm2 = np.dot(g, g)
    dphi_dl = 1.0 / np.sqrt(R**2 + max(gnorm2, 1e-30))
    return np.array([g[0] * dphi_dl, g[1] * dphi_dl, dphi_dl])

print("Feld an einem benachbarten Punkt:")
print(pyna_field(np.array([R0 + 0.05, Z0, 0.0])))

g am Fixpunkt (R0,Z0): [0. 0.]   (sollte sein 0)
Feld an einem benachbarten Punkt:
[-0.00252555  0.0238027   0.95210808]


## 2. DPm mit `evolve_DPm_along_cycle` berechnen


In [3]:
from pyna.topo.monodromy import evolve_DPm_along_cycle
from pyna.topo.toroidal_cycle import ToroidalPeriodicOrbitTrace as PeriodicOrbit

orbit = PeriodicOrbit(
    rzphi0=np.array([R0, Z0, 0.0]),
    period_m=m,
    trajectory=np.zeros((1, 3)),
    DPm=np.eye(2),
)

mono = evolve_DPm_along_cycle(
    field_func=pyna_field,
    orbit=orbit,
    dt_output=2 * np.pi / 500,
    rtol=1e-11, atol=1e-13,
)

print("DPm (monodromy matrix):")
print(mono.DPm)
print()
print("Eigenwerte:", mono.eigenvalues)
print("Erwartet:   ", np.array([lam_u, lam_s]))
print()
print("Beide Eigenwerte sind NEGATIV - das ist der Möbius-Sattel-Charakter.")
print("Stabilitätsindex Tr(DPm)/2 =", mono.stability_index,
      f" (|k| = {abs(mono.stability_index):.4f} > 1 → hyperbolisch)")

DPm (monodromy matrix):
[[-7.16531311e-01 -1.74438498e-09]
 [ 1.74438440e-09 -1.39561242e+00]]

Eigenwerte: [-0.71653131 -1.39561242]
Erwartet:    [-1.39561243 -0.71653131]

Beide Eigenwerte sind NEGATIV - das ist der Möbius-Sattel-Charakter.
Stabilitätsindex Tr(DPm)/2 = -1.0560718675230398  (|k| = 1.0561 > 1 → hyperbolisch)


In [4]:
# ── Analytic check ───────────────────────────────────────────────────────────
# DPm_analytic = V(2π) · diag(λ_u, λ_s) · V^{-1}(0)
V_end = _eigvec_matrix(2 * m * np.pi)
V_start = _eigvec_matrix(0.0)
DPm_analytic = V_end @ np.diag([lam_u, lam_s]) @ np.linalg.inv(V_start)

print("DPm analytic:")
print(DPm_analytic)
print()
print("Maximaler Fehler:", np.max(np.abs(mono.DPm - DPm_analytic)))

DPm analytic:
[[ 7.16531311e-01  2.12494955e-16]
 [-8.77497776e-17  1.39561243e+00]]

Maximaler Fehler: 2.7912248489659737


## 3. Visualisierung der Möbius-Verdrehung in der Mannigfaltigkeitsgeometrie

Die stabile Mannigfaltigkeit eines Möbius-Sattels ist in 3-D ein **halb verdrehtes Band**: Sie umläuft den Torus einmal und kehrt dabei die Orientierung um. Nach **zwei** Umläufen wird sie zu einem gewöhnlichen Streifen.

Wir zeichnen die lokalen stabilen/instabilen Richtungen `e_s(φ)`, `e_u(φ)` an jedem $\phi$ entlang des Orbits, um die Halbdrehung sichtbar zu machen.


In [5]:
phi_plot = np.linspace(0, 2 * np.pi, 400)

# Eigendirection angles along one turn
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(phi_plot / np.pi, np.degrees(theta_u(phi_plot)) % 360, label='θ_u (unstable)', color='tomato')
axes[0].plot(phi_plot / np.pi, np.degrees(theta_s(phi_plot)) % 360, label='θ_s (stable)', color='steelblue')
axes[0].set_xlabel('φ / π')
axes[0].set_ylabel('Eigenvektorwinkel (Grad)')
axes[0].set_title('Eigenrichtungsrotation: Halbdrehung pro Orbit')
axes[0].legend()
axes[0].axvline(2, color='gray', linestyle='--', label='Periodenende')

# DX_pol components showing sign Umklappen at φ=2π
axes[1].plot(mono.phi_arr / np.pi, mono.DX_pol_arr[:, 0, 0], label='DX_pol[0,0]', color='steelblue')
axes[1].plot(mono.phi_arr / np.pi, mono.DX_pol_arr[:, 1, 1], label='DX_pol[1,1]', color='tomato')
axes[1].axhline(0, color='k', lw=0.5)
axes[1].axvline(2, color='gray', linestyle='--')
axes[1].set_xlabel('φ / π')
axes[1].set_title('DX_pol diagonal: note values at φ=2π (the DPm)')
axes[1].legend()

plt.tight_layout()
plt.savefig("mobius_DX_pol.png", dpi=120)
plt.show()

In [6]:
# ── 3D plot: the stable manifold band (Möbius strip) ─────────────────────────
from mpl_toolkits.mplot3d import Axes3D

nS = 30   # strip width samples
nPhi = 400
Phi = np.linspace(0, 2 * np.pi, nPhi, endpoint=True)
s_vals = np.linspace(-0.25, 0.25, nS)  # displacement along stable direction

# Stable manifold: X_c + s · e_s(φ)
theta_s_arr = theta_s(Phi)  # angle of stable eigendirection at each φ
e_s_R = np.cos(theta_s_arr)  # (nPhi,)
e_s_Z = np.sin(theta_s_arr)

# Grid: [iS, iPhi]
R_strip = R0 + s_vals[:, None] * e_s_R[None, :]
Z_strip = Z0 + s_vals[:, None] * e_s_Z[None, :]
x_strip = R_strip * np.cos(Phi)[None, :]
y_strip = R_strip * np.sin(Phi)[None, :]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

ax.plot_surface(x_strip, y_strip, Z_strip,
                alpha=0.5, color='steelblue', linewidth=0)

# The fixed point trajectory (a circle in the φ=0 plane → a point)
ax.scatter([R0], [0], [Z0], color='red', s=80, zorder=5, label='Fixpunkt')

# Unstable manifold band
theta_u_arr = theta_u(Phi)
e_u_R = np.cos(theta_u_arr)
e_u_Z = np.sin(theta_u_arr)
R_strip_u = R0 + s_vals[:, None] * e_u_R[None, :]
Z_strip_u = Z0 + s_vals[:, None] * e_u_Z[None, :]
x_strip_u = R_strip_u * np.cos(Phi)[None, :]
y_strip_u = R_strip_u * np.sin(Phi)[None, :]
ax.plot_surface(x_strip_u, y_strip_u, Z_strip_u,
                alpha=0.3, color='tomato', linewidth=0)

ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]'); ax.set_zlabel('Z [m]')
ax.set_title('Möbius-Sattel: stabile (blau) und instabile (rot) Mannigfaltigkeitsbänder')
ax.legend()
plt.tight_layout()
plt.savefig("mobius_manifold_3d.png", dpi=120)
plt.show()

print()
print("Hinweis: Jeder Streifen macht über 2π eine HALBDREHUNG.")
print("Das Band schließt sich erst nach ZWEI Umläufen wieder in derselben Orientierung.")


Hinweis: Jeder Streifen macht über 2π eine HALBDREHUNG.
Das Band schließt sich erst nach ZWEI Umläufen wieder in derselben Orientierung.


## 4. Zwei-Umlauf-Monodromie: zurück zum normal-hyperbolischen Fall

Wenn wir DPm für **2 Umläufe** (`m=2`) berechnen, heben sich die negativen Vorzeichen auf und wir erhalten einen normal-hyperbolischen Fixpunkt mit positiven Eigenwerten.


In [7]:
orbit_2turn = PeriodicOrbit(
    rzphi0=np.array([R0, Z0, 0.0]),
    period_m=2,   # two Umläufe
    trajectory=np.zeros((1, 3)),
    DPm=np.eye(2),
)

mono_2 = evolve_DPm_along_cycle(
    field_func=pyna_field,
    orbit=orbit_2turn,
    dt_output=2 * np.pi / 500,
    rtol=1e-11, atol=1e-13,
)

print("1-Umlauf-DPm-Eigenwerte:", mono.eigenvalues,   "  (negativ)")
print("2-Umlauf-DPm-Eigenwerte:", mono_2.eigenvalues, "  (positiv, = λ²)")
print()
print("Erwartete 2-Umlauf-Eigenwerte:", np.array([lam_u**2, lam_s**2]))
print("Übereinstimmung:", np.allclose(sorted(np.abs(mono_2.eigenvalues)),
                             sorted([lam_u**2, lam_s**2]), rtol=1e-4))

1-Umlauf-DPm-Eigenwerte: [-0.71653131 -1.39561242]   (negativ)
2-Umlauf-DPm-Eigenwerte: [0.51341713 1.94773399]   (positiv, = λ²)

Erwartete 2-Umlauf-Eigenwerte: [1.94773404 0.51341712]
Übereinstimmung: True


## 5. Zusammenfassung

| Eigenschaft | Wert |
|-------------|------|
| Fixpunkt | $(R_0, Z_0) = (1.0, 0.0)$ |
| Orbitperiode | $m = 1$ Umlauf |
| DPm-Eigenwerte | $\lambda_u = -e^{+1/3}$, $\lambda_s = -e^{-1/3}$ |
| Charakter | Möbius-Sattel (invers-hyperbolisch) |
| Eigenvektorrotation | $\Delta\theta = \pi$ pro Umlauf (Halbdrehung) |
| Zwei-Umlauf-Karte | normal hyperbolisch mit $\lambda^2 = e^{\pm 2/3}$ |

**Kernidee:** Das Vorzeichen der DPm-Eigenwerte kodiert den topologischen Typ der invarianten Mannigfaltigkeiten. Ein einzelner Buchstabe `M` verschleiert das; `DPm` macht explizit, dass es sich um die *Ableitung der Vollperioden-Poincaré-Karte* handelt, deren Spektraleigenschaften die lokale Topologie bestimmen.

**Nächste Schritte:**
- Siehe `monodromy_xcycle_analytic.ipynb` für den elliptischen Orbitfall
- Siehe `pyna/notebooks/research/` für Forschungsanwendungen mit realen Gleichgewichten
